# Decision Tree Classifier Example (Wine Quality Dataset)

Here it is demonstrated how to use the `DecisionTreeClassifier` module from the CMOR-438 library to classify wine quality.
In this example, the Wine Quality dataset is used to train, test, and evaluate the model.

**Goal: Classify wine into three quality tiers (Low, Mid, High) based on physicochemical features.**

The three quality tiers are:
- **Class 0 — Low:** Quality score 3–4
- **Class 1 — Mid:** Quality score 5–6
- **Class 2 — High:** Quality score 7–8

## 1. Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys, os

# Add the algorithm folder and shared helpers to path
NOTEBOOK_DIR = os.path.abspath('')
sys.path.insert(0, NOTEBOOK_DIR)
sys.path.insert(0, os.path.join(NOTEBOOK_DIR, '..', '_shared'))

# Data lives two levels up from the algorithm folder
DATA_DIR = os.path.join(NOTEBOOK_DIR, '..', '..', 'data')
from decision_trees import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

wine = pd.read_csv(os.path.join(DATA_DIR, 'WineQT.csv')).drop(columns=['Id'])
FEATURE_COLS = [c for c in wine.columns if c != 'quality']
print(f"Dataset loaded: {wine.shape[0]} samples, {len(FEATURE_COLS)} features.")
print(f"Features: {FEATURE_COLS}")

## 2. Preprocessing

In [ ]:
X = StandardScaler().fit_transform(wine[FEATURE_COLS].values.astype(float))
y_clf = np.array([0 if q<=4 else (1 if q<=6 else 2) for q in wine['quality'].values])
X_tr, X_te, y_tr, y_te = train_test_split(X, y_clf, test_size=0.2, random_state=42, stratify=y_clf)

print(f"Training samples: {X_tr.shape[0]}  |  Test samples: {X_te.shape[0]}")
print(f"Class distribution: {dict(zip(*np.unique(y_clf, return_counts=True)))}")

## 3. Train

In [ ]:
dtc = DecisionTreeClassifier(criterion='gini', max_depth=8, min_samples_leaf=4)
dtc.fit(X_tr, y_tr)
print(f'Accuracy: {dtc.accuracy(X_te, y_te):.4f}')
print(f'Actual tree depth: {dtc.get_depth()}')

## 4. Results and Visualisation

Two plots are produced:
- **Feature importances** — which wine properties most strongly determine quality
- **Depth vs accuracy sweep** — shows the point where adding depth stops helping

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

idx = np.argsort(dtc.feature_importances_)[::-1]
colors = plt.cm.viridis(np.linspace(0.2, 0.85, len(FEATURE_COLS)))
axes[0].bar(range(len(FEATURE_COLS)), dtc.feature_importances_[idx], color=colors, edgecolor='white')
axes[0].set_xticks(range(len(FEATURE_COLS)))
axes[0].set_xticklabels([FEATURE_COLS[i] for i in idx], rotation=40, ha='right', fontsize=8)
axes[0].set_ylabel('Gini Importance')
axes[0].set_title('Decision Tree - Feature Importances', fontweight='bold')

depths = range(2, 16)
depth_accs = []
for d in depths:
    m = DecisionTreeClassifier(criterion='gini', max_depth=d, min_samples_leaf=4)
    m.fit(X_tr, y_tr)
    depth_accs.append(m.accuracy(X_te, y_te))
axes[1].plot(list(depths), depth_accs, 's-', color='seagreen', lw=1.5, ms=6)
axes[1].axvline(8, color='red', linestyle='--', lw=1.2, label='Chosen depth=8')
axes[1].set_xlabel('Max Depth'); axes[1].set_ylabel('Test Accuracy')
axes[1].set_title('Decision Tree - Depth vs Accuracy', fontweight='bold')
axes[1].legend()
plt.tight_layout(); plt.show()